In [1]:
#imports
import os
import torch
import numpy as np
from torch import nn
from datasets import load_dataset
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    GPT2Tokenizer
)
from transformers.modeling_outputs import SequenceClassifierOutput
from evaluate import load
from model import GPT, GPTConfig
from tokenizer import load_tokenizer  # if used in model.py or checkpoint handling


/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class GPTForSequenceClassification(nn.Module):
    def __init__(self, pretrained_model, num_classes=2, dropout_prob=0.1):
        super().__init__()
        self.pretrained_model = pretrained_model
        self.num_classes = num_classes
        self.hidden_size = pretrained_model.config.n_embd
        self.pad_token_id = pretrained_model.config.pad_token_id

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(self.hidden_size, num_classes, bias=False)
        self.classifier.weight.data.normal_(mean=0.0, std=0.02)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        hidden_states = self.pretrained_model(input_ids)
        logits = self.classifier(hidden_states)

        batch_size, sequence_length = input_ids.shape[:2]
        non_pad_mask = (input_ids != self.pad_token_id).to(logits.device, torch.int32)
        token_indices = torch.arange(input_ids.shape[-1], device=logits.device, dtype=torch.int32)
        last_non_pad_token = (token_indices * non_pad_mask).argmax(-1)
        pooled_logits = logits[torch.arange(batch_size, device=logits.device), last_non_pad_token]

        pooled_logits = self.dropout(pooled_logits)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(pooled_logits, labels)

        return SequenceClassifierOutput(
            loss=loss,
            logits=pooled_logits,
            hidden_states=None,
            attentions=None,
        )


In [3]:
def load_pretrained_model(path, tokenizer, device='cuda'):
    checkpoint = torch.load(path, map_location='cpu')
    model_args = checkpoint.get("model_args", {})

    if "vocab_size" not in model_args:
        model_args["vocab_size"] = tokenizer.vocab_size

    gptconf = GPTConfig(**model_args)
    pretrained_model = GPT(gptconf)
    state_dict = checkpoint["model"]

    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model_dict = pretrained_model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_state_dict)
    pretrained_model.load_state_dict(model_dict)

    # Adjust embedding if vocab size mismatched
    model_vocab_size = pretrained_model.transformer.wte.num_embeddings
    tokenizer_vocab_size = tokenizer.vocab_size
    if tokenizer_vocab_size != model_vocab_size:
        pretrained_model.transformer.wte = torch.nn.Embedding(
            tokenizer_vocab_size,
            pretrained_model.config.n_embd
        )

    pretrained_model.to(device)
    return pretrained_model


In [4]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("glue", "mnli")
train_dataset = dataset["train"]
eval_dataset = dataset["validation_matched"]

def preprocess(example):
    pair = f"Premise: {example['premise']} \nHypothesis: {example['hypothesis']}"
    return tokenizer(
        pair,
        truncation=True,
        padding=True,
        max_length=512,
    )

encoded_train = train_dataset.map(preprocess)
encoded_eval = eval_dataset.map(preprocess)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 9815/9815 [00:03<00:00, 2946.00 examples/s]


In [5]:
accuracy_metric = load("accuracy")
recall_metric = load("recall")
f1_metric = load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "recall": recall_metric.compute(predictions=preds, references=labels, average="macro")["recall"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    }


In [9]:
training_args = TrainingArguments(
    output_dir="./gpt2-mnli-matched-cls",
    evaluation_strategy="steps",
    eval_steps=2000,
    save_strategy="epoch",
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none",
    warmup_ratio=0.1,
)


In [10]:
# devices = {name: "cuda" if torch.cuda.is_available() else "cpu" for name in model_paths}

# for name, path in model_paths.items():
#     print(f"\n==== Loading model: {name} ====")
#     base_model = load_pretrained_model(path, tokenizer, devices[name])
#     base_model.config.pad_token_id = tokenizer.pad_token_id
#     base_model.config.padding_side = tokenizer.padding_side

#     # Print model architecture details only
#     print(f"{name} model config:")
#     print(f"  n_layer: {base_model.config.n_layer}")
#     print(f"  n_embd:  {base_model.config.n_embd}")
#     print(f"  n_head:  {base_model.config.n_head}")


In [ ]:
# model_paths = {
#     "ipa": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt",
#     "normal": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt",
#     "prebuilt": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_medium/ckpt.pt",
# }

# Small models
model_paths = {
    "ipa": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_50k/ckpt.pt",
    "normal": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_50k/ckpt.pt",
    # "prebuilt": "/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_50k/ckpt.pt",
}

devices = {name: "cuda" if torch.cuda.is_available() else "cpu" for name in model_paths}
results = {}

for name, path in model_paths.items():
    print(f"\n==== Training and evaluating model: {name} ====")
    base_model = load_pretrained_model(path, tokenizer, devices[name])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model, num_classes=3).to(devices[name])

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_train,
        eval_dataset=encoded_eval,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_result = trainer.evaluate()
    results[name] = eval_result

# Print results summary
print("\n==== Evaluation Results Summary ====")
for name, metrics in results.items():
    print(f"\n{name} Results:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")



==== Training and evaluating model: ipa ====
number of parameters: 123.35M


/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,Recall,F1
2000,1.065500,1.061479,0.424758,0.435172,0.397293
4000,0.992200,0.974955,0.527866,0.529379,0.525474
6000,0.924700,0.928782,0.574733,0.573428,0.569317
8000,0.908600,0.911526,0.585023,0.587578,0.585595
10000,0.880200,0.857900,0.614060,0.612271,0.612383
12000,0.877800,0.852860,0.610596,0.609712,0.609845
14000,0.847700,0.836928,0.625981,0.625314,0.625291
16000,0.862900,0.838319,0.619766,0.619705,0.619585
18000,0.837300,0.830503,0.625369,0.624763,0.623911
20000,0.832200,0.813039,0.633418,0.631375,0.631287



==== Training and evaluating model: normal ====
number of parameters: 123.35M


/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,Recall,F1
2000,1.065700,1.053595,0.427101,0.438211,0.395694
4000,0.979800,0.969287,0.536016,0.541703,0.530865
6000,0.913000,0.909557,0.581966,0.581243,0.578364
8000,0.901400,0.889145,0.596638,0.598284,0.596778
10000,0.876900,0.856752,0.610188,0.607431,0.606012
12000,0.866400,0.847641,0.619256,0.618657,0.618765
14000,0.844400,0.847006,0.613347,0.615490,0.613072
16000,0.861700,0.833531,0.630158,0.630177,0.629992
18000,0.831200,0.833102,0.624554,0.623596,0.622710
20000,0.830600,0.814716,0.635762,0.632614,0.632540


In [ ]:
trainer.evaluate()